# Module 6 Labs 6-3 & 6-4: LoRA Fine-Tuning

**This notebook requires a GPU runtime.** Go to `Runtime > Change runtime type > T4 GPU`.

Labs 6-1 and 6-2 run on CPU and can be done locally.


In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets evaluate peft torch transformers trl


## Lab 6-3: LLaMA LoRA Fine-Tuning with Unsloth

Run each part in sequence. Parts B-E depend on the model loaded in Part A.


### Part A: Load Model & Apply LoRA


In [ ]:
from unsloth import FastLanguageModel

# Load LLaMA-3.2 1B (4-bit quantized for memory efficiency)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()


### Part B: Test Before Fine-Tuning


In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Explain what LoRA is in simple terms.",
    "Write a Python function that reverses a string.",
    "What are the benefits of parameter-efficient finetuning?",
]

print("=== BEFORE Finetuning ===\n")
for prompt in test_prompts:
    inputs = tokenizer(
        f"### Instruction:\n{prompt}\n\n### Response:\n",
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=200, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")
    print("-" * 60)


### Part C: Fine-Tune


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset("mlabonne/FineTome-100k", split="train[:5000]")

def format_prompt(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

dataset = dataset.map(format_prompt)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")


### Part D: Test After Fine-Tuning


In [ ]:
FastLanguageModel.for_inference(model)

print("\n=== AFTER Finetuning ===\n")
for prompt in test_prompts:
    inputs = tokenizer(
        f"### Instruction:\n{prompt}\n\n### Response:\n",
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=200, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}\n")
    print("-" * 60)


### Part E: Save LoRA Adapter


In [ ]:
import os

model.save_pretrained("lora_adapter")

adapter_size = sum(
    os.path.getsize(os.path.join("lora_adapter", f))
    for f in os.listdir("lora_adapter")
)
print(f"LoRA adapter size: {adapter_size / 1e6:.1f} MB")
print(f"(Compare to full model: ~2500 MB in FP16)")


---
## Lab 6-4: LoRA Rank Sweep (BERT + SST-2)

This lab compares different LoRA ranks on a smaller model (BERT) to observe the accuracy/parameter trade-off.


In [ ]:

import json

# We will use BERT + SST-2 for this experiment since it is fast to train
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import evaluate

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
dataset = load_dataset("glue", "sst2")

def tokenize_fn(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Ranks to test
ranks = [4, 8, 16, 32]
results = {}

for r in ranks:
    print(f"\n{'='*50}")
    print(f"Training with LoRA rank = {r}")
    print(f"{'='*50}")

    # Load fresh base model
    base_model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2
    )

    # Apply LoRA
    lora_config = LoraConfig(
        r=r,
        lora_alpha=r,  # Keep alpha = r for fair comparison
        target_modules=["query", "value"],
        lora_dropout=0.1,
        bias="none",
        task_type="SEQ_CLS",
    )
    lora_model = get_peft_model(base_model, lora_config)

    # Count params
    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in lora_model.parameters())

    # Train
    training_args = TrainingArguments(
        output_dir=f"./lora_rank_{r}",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        warmup_steps=100,
        weight_decay=0.01,
        learning_rate=2e-4,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    results[r] = {
        "rank": r,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_fraction": trainable / total,
        "accuracy": eval_results["eval_accuracy"],
    }

    print(f"Rank {r}: {trainable:,} params ({trainable/total:.4%}), "
          f"Accuracy: {eval_results['eval_accuracy']:.4f}")

# Print summary table
print(f"\n{'='*70}")
print(f"{'Rank':>6} | {'Trainable':>12} | {'Fraction':>10} | {'Accuracy':>10}")
print(f"{'-'*6}-+-{'-'*12}-+-{'-'*10}-+-{'-'*10}")
for r, res in sorted(results.items()):
    print(f"{res['rank']:>6} | {res['trainable_params']:>12,} | "
          f"{res['trainable_fraction']:>9.4%} | {res['accuracy']:>9.4f}")

# Save results
with open("rank_sweep_results.json", "w") as f:
    json.dump(results, f, indent=2)

# ============================================================
# Part B
# ============================================================

import matplotlib.pyplot as plt

ranks_list = sorted(results.keys())
params_list = [results[r]["trainable_params"] for r in ranks_list]
acc_list = [results[r]["accuracy"] for r in ranks_list]

fig, ax1 = plt.subplots(figsize=(8, 5))

color1 = "#2196F3"
color2 = "#FF5722"

ax1.set_xlabel("LoRA Rank", fontsize=12)
ax1.set_ylabel("Accuracy", fontsize=12, color=color1)
ax1.plot(ranks_list, acc_list, "o-", color=color1, linewidth=2, markersize=8, label="Accuracy")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_ylim([0.80, 0.95])

ax2 = ax1.twinx()
ax2.set_ylabel("Trainable Parameters", fontsize=12, color=color2)
ax2.bar(ranks_list, params_list, alpha=0.3, color=color2, width=2, label="Params")
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("LoRA Rank vs Accuracy and Parameter Count (BERT + SST-2)", fontsize=13)
fig.tight_layout()
plt.savefig("rank_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

print("Plot saved to rank_vs_accuracy.png")

